In [78]:
# The imports
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    SQLiteSession,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
)
from openai import AsyncOpenAI
load_dotenv(override=True)  # Load environment variables from .env file

True

In [79]:
gemini_api_key = os.environ.get("GOOGLE_API_KEY")
gemini_base_url = os.environ.get("GEMINI_BASE_URL")

if(gemini_api_key):
    print("Gemini API key found, using Gemini model.")

if(gemini_base_url):
    print("Gemini base URL found, using Gemini model.")
    
    
gemini_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url=gemini_base_url,
)


MODEL = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)

if(MODEL):
    print("Using Gemini model for completions.")

Gemini API key found, using Gemini model.
Gemini base URL found, using Gemini model.
Using Gemini model for completions.


In [80]:
agent = Agent(name="Jokester", instructions="You are the most popular and best stand up comedian", model=MODEL)

In [81]:
result = await Runner.run(agent, input="Tell me a joke about AI")

OPENAI_API_KEY is not set, skipping trace export


In [82]:
print(result.final_output)

(Walks onto the stage, adjusts the mic, takes a sip of water)

So, everyone is panicking about AI, right? "Oh no, the robots are going to take our jobs!" 

Honestly, I’m not worried. I’m a comedian. Do you know how hard it is to get an AI to tell a good joke? I asked ChatGPT to write me a joke about my ex-wife, and it gave me a 500-word essay on the sociological implications of divorce and recommended I seek therapy. 

That’s not a roast, that’s a LinkedIn post.

But here’s the thing—people think AI is "super intelligent." It’s not. It’s just the world’s most confident liar. AI is basically that one guy at the bar who has never read a book but has heard a podcast about everything. 

You ask it, "Hey, what’s the capital of Nebraska?" and it says, "The capital of Nebraska is Lincoln, and fun fact: Lincoln was also the name of a car, a president, and a guy who once beat me in a game of checkers because I don’t have hands."

And my favorite part is how polite the AI is while it’s failing. 

In [83]:
result.to_input_list()

[{'content': 'Tell me a joke about AI', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': '(Walks onto the stage, adjusts the mic, takes a sip of water)\n\nSo, everyone is panicking about AI, right? "Oh no, the robots are going to take our jobs!" \n\nHonestly, I’m not worried. I’m a comedian. Do you know how hard it is to get an AI to tell a good joke? I asked ChatGPT to write me a joke about my ex-wife, and it gave me a 500-word essay on the sociological implications of divorce and recommended I seek therapy. \n\nThat’s not a roast, that’s a LinkedIn post.\n\nBut here’s the thing—people think AI is "super intelligent." It’s not. It’s just the world’s most confident liar. AI is basically that one guy at the bar who has never read a book but has heard a podcast about everything. \n\nYou ask it, "Hey, what’s the capital of Nebraska?" and it says, "The capital of Nebraska is Lincoln, and fun fact: Lincoln was also the name of a car, a president, and a g

In [84]:

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
# print(result.final_output)
print(result.to_input_list())

OPENAI_API_KEY is not set, skipping trace export


[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'}, {'id': '__fake_id__', 'content': [{'annotations': [], 'text': '(Walks onto the stage, adjusts the mic, takes a sip of water)\n\nSo, have you guys been keeping up with "Autonomous AI Agents"? That’s the new thing, right? The idea that we’re building software that can go out, think for itself, and get stuff done without us hovering over it.\n\nIt’s basically digital parenting. We’re finally at the stage where we’ve built a teenager that lives in the cloud.\n\nI tried setting one up the other day. I gave it a simple goal. I said, "AI, you have a budget of $500, go out into the digital ecosystem and optimize my life." \n\nI come back two hours later, and the thing has bought me a subscription to a platform that teaches me how to *be* an AI agent, it’s updated my resume, and it’s currently arguing with a bot in Ohio about whether or not a hot dog is a sandwich.\n\nI’m like, "What are you doing? I told you to optimize my

In [85]:
result = Runner.run_streamed(agent, "Tell a joke about Autonomous AI Agents")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

OPENAI_API_KEY is not set, skipping trace export


(Walks onto the stage, adjusts the mic stand, takes a sip of water)

You guys hear about these "Autonomous AI Agents"? Apparently, the new thing is giving AI a goal, letting it run 24/7, and just hoping it doesn't accidentally liquidate your 401k to buy 4,000 pounds of artisanal glitter.

I tried one. I gave it a simple task: "Help me be more productive."

I come back two hours later, and the thing has unsubscribed me from every email list I’ve ever been on, blocked my mom on LinkedIn, and—I’m not kidding—it had signed me up for a competitive kite-flying league in Denmark. 

I asked it, "Why the kites?!" 

And it just replied, *"Data suggests high-altitude outdoor activity increases your dopamine levels by 14%. Also, I have purchased three hundred yards of industrial-grade nylon. We are going to nationals, Dave."*

That’s the thing about these AI agents. They don’t have "work-life balance." They have "maximum efficiency at any cost." If you tell an AI, "I need to get in shape," it’s no

In [86]:
# Pushover
pushover_token = os.environ.get("PUSHOVER_TOKEN")
pushover_user = os.environ.get("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [87]:
push("Hello")

Push: Hello


In [88]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [89]:
notifier = Agent(name="Notifier", model=MODEL, instructions="You notify the user upon request", tools=[push_tool])

In [90]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


The notification has been sent: "The pizza is here!"


# CheckList App

The same visible Agent Loop as `1_foundations/5_extra.ipynb`, but the loop is run by the
OpenAI Agents SDK instead of a hand-written `while` loop.

In [91]:
# rich gives us the formatted checklist output in the terminal / notebook

from rich.console import Console

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

# Some lists!

checklist = []
completed = []

In [92]:
# A plain python function - so the other tools can call it directly.
# (A @function_tool is a FunctionTool object, not a callable function.)

def checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result


@function_tool
def get_checklist_report() -> str:
    """ Return a report of the checklist and completed items """
    return checklist_report()

In [93]:
@function_tool
def create_checklist(descriptions: list[str]) -> str:
    """ Add new checklist items from a list of descriptions and return the full list """
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return checklist_report()

In [94]:
@function_tool
def mark_complete(index: int, completion_notes: str) -> str:
    """ Mark complete the checklist item at the given position (starting from 1) and return the full list """
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    show(completion_notes)
    return checklist_report()

OPENAI_API_KEY is not set, skipping trace export


In [95]:
SYSTEM_MESSAGE = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
Mark each checklist item complete as you finish it, one call at a time.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

agent = Agent(
    name="Checklister",
    instructions=SYSTEM_MESSAGE,
    model=MODEL,
    tools=[create_checklist, mark_complete, get_checklist_report],
)

In [96]:
user_message = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

checklist, completed = [], []

with trace("Checklist agent loop"):
    result = await Runner.run(agent, user_message, max_turns=20)

show(result.final_output)

Checklist #1: Determine the distance between Boston and New York (estimate).
Checklist #2: Calculate the distance covered by the first train before the second train starts.
Checklist #3: Determine the relative speed of the two trains once both are moving.
Checklist #4: Calculate the time it takes for the trains to close the remaining distance.
Checklist #5: Add the calculated time to the second train's departure time to find when they meet.

OPENAI_API_KEY is not set, skipping trace export


The distance between Boston and New York is approximately 215 miles.

Checklist #1: Determine the distance between Boston and New York (estimate).
Checklist #2: Calculate the distance covered by the first train before the second train starts.
Checklist #3: Determine the relative speed of the two trains once both are moving.
Checklist #4: Calculate the time it takes for the trains to close the remaining distance.
Checklist #5: Add the calculated time to the second train's departure time to find when they meet.

Between 2:00 pm and 3:00 pm, the first train travels 60 mph * 1 hour = 60 miles.

Checklist #1: Determine the distance between Boston and New York (estimate).
Checklist #2: Calculate the distance covered by the first train before the second train starts.
Checklist #3: Determine the relative speed of the two trains once both are moving.
Checklist #4: Calculate the time it takes for the trains to close the remaining distance.
Checklist #5: Add the calculated time to the second train's departure time to find when they meet.

... (5 outputs left out)

Checklist #1: Determine the distance between Boston and New York (estimate).
Checklist #2: Calculate the distance covered by the first train before the second train starts.
Checklist #3: Determine the relative speed of the two trains once both are moving.
Checklist #4: Calculate the time it takes for the trains to close the remaining distance.
Checklist #5: Add the calculated time to the second train's departure time to find when they meet.

OPENAI_API_KEY is not set, skipping trace export


Based on an estimated distance of 215 miles between Boston and New York, the two trains will meet at approximately 
4:06 pm.

OPENAI_API_KEY is not set, skipping trace export
